In [1]:
import sys
sys.path.append('../scripts')
from robot import *
from scipy.stats import multivariate_normal

In [3]:
class Particle:
    def __init__(self,init_pose):
        self.pose = init_pose

    def motion_update(self,nu,omega,time,noise_rate_pdf):
        ns = noise_rate_pdf.rvs()
        noised_nu = nu + ns[0]*math.sqrt(abs(nu)/time) + ns[1]*math.sqrt(abs(omega)/time)
        noised_omega = omega + ns[2]*math.sqrt(abs(nu)/time) + ns[3]*math.sqrt(abs(omega)/time)
        self.pose = IdealRobot.state_transition(noised_nu,noised_omega,time,self.pose)

In [4]:
class Mcl:
    def __init__(self,init_pose,num,motion_noise_stds):
        self.particles = [Particle(init_pose) for i in range(num)]

        v = motion_noise_stds
        c = np.diag([v["nn"]**2,v["no"]**2,v["on"]**2,v["oo"]**2])
        self.motion_noise_rate_pdf = multivariate_normal(cov=c)

    def motion_update(self,nu,omega,time):
        for p in self.particles:p.motion_update(nu,omega,time,self.motion_noise_rate_pdf)

    def draw(self,ax,elems):
        xs = [p.pose[0] for p in self.particles]
        ys = [p.pose[1] for p in self.particles]
        vxs = [math.cos(p.pose[2]) for p in self.particles]
        vys = [math.sin(p.pose[2]) for p in self.particles]
        elems.append(ax.quiver(xs,ys,vxs,vys,color="blue",alpha=0.5))

In [2]:
class EstimationAgent(Agent):
    def __init__(self,time_interval,nu,omega,estimator):
        super().__init__(nu,omega)
        self.estimator = estimator
        self.time_interval = time_interval

        self.prev_nu = 0.0
        self.prev_omega = 0.0

    def decision(self,observation=None):
        self.estimator.motion_update(self.prev_nu,self.prev_omega,self.time_interval)
        self.prev_nu,self.prev_omega = self.nu,self.omega
        return self.nu,self.omega

    def draw(self,ax,elems):
        self.estimator.draw(ax,elems)

In [10]:
initial_pose = np.array([0,0,0]).T
estimator = Mcl(initial_pose,100,motion_noise_stds={"nn":0.01, "no":0.02, "on":0.03, "oo":0.04})
a = EstimationAgent(0.1,0.2,10.0/180*math.pi,estimator)
estimator.motion_update(0.2,10.0/180*math.pi,0.1)
for p in estimator.particles:
    print(p.pose)

[1.21349824e-02 6.41249910e-05 1.05685187e-02]
[0.022751   0.00016516 0.01451831]
[0.02063835 0.00011832 0.01146607]
[0.01603648 0.00014854 0.01852433]
[0.01777177 0.00012208 0.01373896]
[0.02442335 0.00035746 0.02926979]
[0.01931132 0.00027931 0.02892556]
[0.01953723 0.00023138 0.02368447]
[0.01935658 0.00029944 0.03093652]
[0.01839196 0.00022558 0.02452888]
[0.01714294 0.00025199 0.02939635]
[0.02339173 0.00028879 0.02469058]
[0.01754165 0.00016979 0.01935809]
[0.01902333 0.00021716 0.02282966]
[0.01887736 0.00013346 0.01413914]
[0.01699708 0.000159   0.01870835]
[0.01837875 0.00011461 0.01247159]
[0.02588158 0.00041413 0.031999  ]
[0.01957462 0.00019881 0.020312  ]
[0.02010114 0.00012319 0.01225641]
[0.01808058 0.00020158 0.02229686]
[0.0186552  0.00011856 0.0127105 ]
[0.02789523 0.00012049 0.00863882]
[0.0177602  0.00031553 0.03552837]
[0.02470594 0.00025346 0.02051735]
[0.02154503 0.00013754 0.01276738]
[0.01962783 0.00017957 0.01829678]
[0.01488716 0.000104   0.01397215]
[0.01040

In [ ]:
def trial(motion_noise_stds):
    time_interval = 0.1
    world=World(30,time_interval)

    initial_pose = np.array([0,0,0]).T
    estimator = Mcl(initial_pose,100,motion_noise_stds)
    circling = EstimationAgent(time_interval,0.2,10.0/180*math.pi,estimator)
    r = Robot(initial_pose,sensor=None,agent=circling,color="red")
    world.append(r)

    return world.draw()

trial({"nn":0.01, "no":0.02, "on":0.03, "oo":0.04})